## Creating a baseline for BKT

## Install pyBKT

In [4]:
!git clone https://github.com/CAHLR/pyBKT.git
%cd pyBKT
!pip install .

Cloning into 'pyBKT'...
remote: Enumerating objects: 4655, done.
remote: Counting objects: 100% (350/350), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 4655 (delta 250), reused 292 (delta 224), pack-reused 4305 (from 1)
Receiving objects: 100% (4655/4655), 3.31 MiB | 17.82 MiB/s, done.
Resolving deltas: 100% (2224/2224), done.
/content/pyBKT
Processing /content/pyBKT
  Preparing metadata (setup.py) ... done
  Created wheel for pyBKT: filename=pyBKT-1.4.3-cp312-cp312-linux_x86_64.whl size=1130718 sha256=1ca77c6dcb1bcb3733be5ac09a459afcac669243bd686a0a62fe23048d85489c
  Stored in directory: /tmp/pip-ephem-wheel-cache-xg2mcatx/wheels/cb/59/14/60b0958d188ae739402b0b53460273f73d657af0856eada58c
Successfully built pyBKT
  Attempting uninstall: pyBKT
    Found existing installation: pyBKT 1.4.3
    Uninstalling pyBKT-1.4.3:
      Successfully uninstalled pyBKT-1.4.3


## Import dataset

In [1]:
import os
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

DATA_FILE = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv"

df = pd.read_csv(DATA_FILE, encoding="latin1")

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print(df[["user_id", "problem_id", "skill_id", "correct"]].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_52355/3404439239.py:9: DtypeWarning: Columns (0: skill_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_FILE, encoding="latin1")


Shape: (525534, 30)
Columns:
['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']

First 5 rows:
   order_id  assignment_id  user_id  assistment_id  problem_id  original  \
0  33022537         277618    64525          33139       51424         1   
1  33022709         277618    64525          33150       51435         1   
2  35450204         220674    70363          33159       51444         1   
3  35450295         220674    70363          33110       51395         1   
4  35450311         220674    70363          33196       51481         1   

   correct  attempt_count  ms_first_response tutor_m

## Prepare dataset for the model

In [5]:
# Create clean BKT dataset using skill_id as the skill identifier

bkt_df = df[
    ["order_id", "user_id", "skill_id", "correct"]
].copy()

# Remove only interactions with no skill ID
bkt_df = bkt_df.dropna(subset=["skill_id"])

# Convert skill IDs from floats (e.g. 37.0) to strings (e.g. "37")
bkt_df["skill_id"] = bkt_df["skill_id"].astype(int).astype(str)

# Make sure correct is numeric 0/1
bkt_df["correct"] = pd.to_numeric(
    bkt_df["correct"], errors="coerce"
)

bkt_df = bkt_df[
    bkt_df["correct"].isin([0, 1])
].copy()

# Preserve the original interaction order
bkt_df = bkt_df.sort_values(
    ["user_id", "order_id"]
).reset_index(drop=True)

print("BKT dataset shape:", bkt_df.shape)
print("Students:", bkt_df["user_id"].nunique())
print("Skills:", bkt_df["skill_id"].nunique())

print("\nFirst rows:")
print(bkt_df.head())

BKT dataset shape: (459208, 4)
Students: 4163
Skills: 123

First rows:
   order_id  user_id skill_id  correct
0  21617623       14        2        0
1  21617623       14       37        0
2  21617623       14       70        0
3  21617632       14        2        1
4  21617632       14       37        1


## Create the 80/20 student split

In [6]:
import numpy as np

## Random seed fixes the exact random congifuration
RANDOM_SEED = 42
## 80% training, 20% testing
TRAIN_RATIO = 0.8

rng = np.random.default_rng(RANDOM_SEED)

students = bkt_df["user_id"].unique()
rng.shuffle(students)

split_idx = int(len(students) * TRAIN_RATIO)

train_students = students[:split_idx]
test_students = students[split_idx:]

train_df = bkt_df[
    bkt_df["user_id"].isin(train_students)
].copy()

test_df = bkt_df[
    bkt_df["user_id"].isin(test_students)
].copy()

print("Total students:", len(students))
print("Training students:", len(train_students))
print("Test students:", len(test_students))

print("\nTraining interactions:", len(train_df))
print("Test interactions:", len(test_df))

Total students: 4163
Training students: 3330
Test students: 833

Training interactions: 378834
Test interactions: 80374


## Verify no students overlap

In [7]:
train_ids = set(train_df["user_id"])
test_ids = set(test_df["user_id"])

print("Overlapping students:", len(train_ids & test_ids))

Overlapping students: 0


## Format data for pyBKT

In [8]:
# Prepare data for pyBKT

bkt_train = train_df[
    ["order_id", "user_id", "skill_id", "correct"]
].copy()

bkt_test = test_df[
    ["order_id", "user_id", "skill_id", "correct"]
].copy()

# pyBKT expects the interaction order to be represented consistently
bkt_train = bkt_train.sort_values(
    ["user_id", "order_id"]
).reset_index(drop=True)

bkt_test = bkt_test.sort_values(
    ["user_id", "order_id"]
).reset_index(drop=True)

print("Training shape:", bkt_train.shape)
print("Test shape:", bkt_test.shape)
print("\nTraining:")
print(bkt_train.head())
print("\nTest:")
print(bkt_test.head())

Training shape: (378834, 4)
Test shape: (80374, 4)

Training:
   order_id  user_id skill_id  correct
0  21617623       14        2        0
1  21617623       14       37        0
2  21617623       14       70        0
3  21617632       14        2        1
4  21617632       14       37        1

Test:
   order_id  user_id skill_id  correct
0  28025073    54318      312        0
1  28025106    54318      312        1
2  28025109    54318      312        1
3  28025112    54318      312        1
4  28025127    54318      312        1


## Fit the model

In [9]:
from pyBKT.models import Model
import time

start = time.time()

bkt_model = Model(
    seed=42,
    num_fits=1,
    parallel=True
)

bkt_model.fit(
    data=bkt_train,
    defaults={
        "order_id": "order_id",
        "student_id": "user_id",
        "skill_name": "skill_id",
        "correct": "correct"
    }
)

elapsed = time.time() - start

print(f"BKT training time: {elapsed:.2f} seconds")
print("BKT fitting complete.")

BKT training time: 1.61 seconds
BKT fitting complete.


## Get AUC

In [12]:
auc = bkt_model.evaluate(
    data=bkt_test,
    metric="auc"
)

print("BKT Test AUC:", auc)

BKT Test AUC: 0.7945034720125428


## Model Comparison — ASSISTments 2009

| Model | Dataset | Key Settings | Test AUC |
|:---|:---|:---|---:|
| **BKT** | ASSISTments 2009 (80/20 split) | 123 skills, `num_fits=1`, C++ backend, parallel training | **0.7945** |
| **SAKT** | ASSISTments 2009 (80/20 split) | Sequence length = 100, 5 attention heads, dropout = 0.2, batch size = 128, 100 epochs, Adam, best LR = 0.0001, d = 125 | **0.8098** |

### Results

- **BKT Test AUC:** 0.7945
- **SAKT Test AUC:** 0.8098
- **SAKT − BKT:** +0.0153 AUC

**SAKT currently outperforms BKT by 0.0153 AUC on the ASSISTments 2009 test set.**